In [1]:
import sys, os
sys.path.append(os.path.abspath("../../"))

# Load shared utilities for attention-from-scratch analysis
from scripts.attention_from_scratch_utils import *

# Set reproducible seed
set_seed(42)

print("Environment ready.")
print("Device:", device)

Environment ready.
Device: mps


# 03b — Learned Positional Embeddings

Transformers do not naturally understand the order of a sequence.  
In this notebook, we explore how **learned positional embeddings** inject order
information into token embeddings so the model can reason about *where* each token appears.

### What we'll explore
- Why positional information is necessary
- How learned positional embeddings work
- Visualizing learned positional vectors

There are two main ways to represent position in a Transformer —  
**learned positional embeddings** and **sinusoidal positional encodings**.  
In this notebook, we focus on understanding **learned positional embeddings**:
how they are stored, how they are indexed, and how they combine with token embeddings.

# What Is a Learned Positional Embedding?

A **learned positional embedding** is simply a vector that the model learns to
represent each position in a sequence.

Instead of using smooth mathematical waves, the model stores a **lookup table**:

- Position 0 → learned vector  
- Position 1 → learned vector  
- Position 2 → learned vector  
- … and so on …

Each vector is shaped like a token embedding (e.g., 512-dimensional).

These vectors start out random and are trained by gradient descent just like
any other parameter.

---

# Key Properties of Learned Positional Embeddings

### **1. One vector per position**
The model has a matrix like:

(max_seq_len, d_model)

Each row corresponds to a position:  
row 0 → position 0, row 1 → position 1, etc.

### **2. No fixed structure**
There are no waves or formulas.  
The model is free to learn whatever representation works best for the data.

### **3. Fully trainable**
During training, the positional embedding vectors update to optimize:

- word order understanding  
- syntactic patterns  
- long-range dependencies  
- whatever the task requires  

The result is a flexible positional representation that adapts to the training distribution.

---

# Everyday Intuition: Why Learn Position Instead of Using Waves?

Think of a learned positional embedding like giving the model a **notebook** where
it gets to write down what each position *tends* to mean in real text.

Instead of hand-crafted waves, the model discovers patterns such as:

- Position **0** often starts a sentence  
- Position **1** often introduces the subject  
- Middle positions usually contain the “meat” of the sentence  
- Later positions often close the thought or finish a phrase  

Over millions of training examples, the model notices these regularities and
adjusts each positional vector to reflect them.

In other words:

- **Early positions** learn “sentence beginning” features  
- **Middle positions** learn “inside the sentence” features  
- **Later positions** learn “ending” features  

Because each position develops its own learned character, every spot in the
sequence ends up with a **distinct, data-driven signature** that helps the
Transformer understand where a token is.

---

# Our Simple 8-Dimensional Learned Positional Embedding

To make this concrete, we’ll use **8 dimensions** instead of 512+.

A learned positional embedding matrix for 10 positions would look like:

| **Position** | **Learned 8-D Vector**               |
|--------------|---------------------------------------|
| 0            | `[0.12, -0.44, …]`                    |
| 1            | `[0.98,  0.02, …]`                    |
| 2            | `[-0.31, 0.55, …]`                    |
| 3            | `[0.77, -0.12, …]`                    |
| …            | `…`                                   |

Each position’s vector is **independent** — the model learns them all.

### Why do these vectors look random at first?

Because they *are* random at initialization.

During training, they gradually take on structure that helps the model understand:

- sentence beginnings  
- middles  
- ends  
- recurring syntactic patterns  

This structure is discovered by the model, not imposed.

---

# Injecting Order Into Embeddings

Learned positional embeddings are added directly to token embeddings:

final_embedding = token_embedding + positional_embedding[position]

This enriched embedding tells the model:

- **what** the token represents  
- **where** it appears in the sequence  

This positional signal flows through every attention head,
giving the Transformer a learned understanding of order.

---

# The Learned Positional Embedding Matrix

There is **no formula**, but structurally it looks like this:

Let:

- `pos` = token position (0, 1, 2, …)
- `d_model` = embedding dimension (e.g., 512)
- `P` = learned positional embedding matrix of shape `(max_seq_len, d_model)`

Then the positional embedding for position `pos` is simply:

X[pos] = TokenEmbed[token_id] + P[pos]

This is a lookup, not a computation.

---

# What Learning Does

During training, gradients adjust each row of `P` based on:

- where the model needs global position awareness  
- where it needs local attention patterns  
- how position affects syntax and meaning  

This allows the model to learn:

- subtle distinctions in early vs. late positions  
- how far apart tokens tend to interact  
- patterns unique to the dataset  

Unlike sinusoids, learned embeddings can model **arbitrary patterns** of position.

---

# A Simple, Easy-to-Follow Example

Let’s walk through a tiny example using a model with:

- **8-dimensional embeddings**
- **10 possible positions**
- We want the embedding for **position = 3**

Assume the learned matrix `P` looks like:

| Position | Embedding (8D)                 |
|----------|--------------------------------|
| 0        | `[0.10, -0.20, …]`             |
| 1        | `[0.05, -0.01, …]`             |
| 2        | `[0.40,  0.12, …]`             |
| 3        | `[-0.03, 0.77, …]`             |
| …        | `…`                            |

### Step 1 — Token embedding lookup

Suppose the token embedding is:

[0.50, -0.10, 0.33, -0.22, …]

### Step 2 — Positional embedding lookup

Position 3 → `P[3] = [-0.03, 0.77, …]`

### Step 3 — Add them

final = token + P[3]

Dimension-wise:

| Dim | Token | PosEmb | Sum |
|-----|--------|--------|------|
| 0   | 0.50   | -0.03  | 0.47 |
| 1   | -0.10  | 0.77   | 0.67 |
| …   | …      | …      | …    |

This forms the final embedding the Transformer uses.

---

# Why This Matters

This example shows:

- Each position has its own **trainable vector**
- The model is free to learn **any representation** for each position
- Order is injected directly into the embedding space
- These representations adapt to the task and dataset

This is the core idea behind learned positional embeddings.

---

# How Learned Embeddings Integrate Into a Transformer

Positional embeddings are added **before** attention:

(tokens → token embedding) + (positions → learned embedding)
→ combined representation → self-attention → …

This gives the model a coordinate system that it optimizes for representing order.

---

# Summary — The Intuition Behind Learned Positional Embeddings

- Transformers do not understand order by default  
- Learned positional embeddings assign each position a **trainable vector**  
- There is **no formula** — the model discovers useful patterns  
- The representation adapts to the task and dataset  
- Token + position = full input representation  
- These learned signals guide attention to understand sequence structure  

By the end of this section, you’ll clearly understand how Transformers learn to represent **where** tokens are in a sequence using trainable positional vectors.

In [4]:
text = (
    "Alice was beginning to get very tired of sitting by her sister on the bank, "
    "and of having nothing to do. Once or twice she had peeped into the book her "
    "sister was reading, but it had no pictures or conversations in it, “and what "
    "is the use of a book,” thought Alice, “without pictures or conversation?” "
    "So she was considering in her own mind, as well as she could, whether the "
    "pleasure of making a daisy-chain would be worth the trouble of getting up "
    "and picking the daisies, when suddenly a White Rabbit with pink eyes ran close "
    "by her. There was nothing so very remarkable in that; nor did Alice think it "
    "so very much out of the way to hear the Rabbit say to itself, “Oh dear! Oh dear! "
    "I shall be too late!” "
    "But when the Rabbit actually took a watch out of its waistcoat-pocket, and "
    "looked at it, and then hurried on, Alice started to her feet, for it flashed "
    "across her mind that she had never before seen a rabbit with either a "
    "waistcoat-pocket, or a watch to take out of it."
)

# Simple whitespace tokenization
tokens = text.split()

# Count them
num_tokens = len(tokens)
d_model = 8

tokens[:10], num_tokens

(['Alice',
  'was',
  'beginning',
  'to',
  'get',
  'very',
  'tired',
  'of',
  'sitting',
  'by'],
 188)

## Next Step: Generate Fake Token Embeddings

We’ll now create synthetic token embeddings for the tokens in our text and combine them with their positional encodings.  
This will show us **exactly** what the Transformer receives as input before computing attention.

In [5]:
import torch
import pandas as pd
from IPython.display import display, HTML

# -------------------------------------------
# Use ALL tokens
# -------------------------------------------
n = len(tokens)              # number of tokens
selected_tokens = tokens     # use the full list

# -------------------------------------------
# Token Embeddings (fake for demo)
# -------------------------------------------
torch.manual_seed(0)
fake_token_embeds = torch.randn(n, d_model)

df_tokens = pd.DataFrame(
    fake_token_embeds.numpy().round(4),
    columns=[f"dim_{i}" for i in range(d_model)]
)

df_tokens.insert(0, "token", selected_tokens)
df_tokens.insert(1, "position", range(n))

# -------------------------------------------
# Display table
# -------------------------------------------
display(HTML("<h3>1) Token Embeddings (learned via training)</h3>"))
display(df_tokens)

,token,position,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7
0,Alice,0,-1.1258,-1.1524,-0.2506,-0.4339,0.8487,0.6920,-0.3160,-2.1152
1,was,1,0.3223,-1.2633,0.3500,0.3081,0.1198,1.2377,1.1168,-0.2473
2,beginning,2,-1.3527,-1.6959,0.5667,0.7935,0.5988,-1.5551,-0.3414,1.8530
3,to,3,0.7502,-0.5855,-0.1734,0.1835,1.3894,1.5863,0.9463,-0.8437
4,get,4,-0.6136,0.0316,-0.4927,0.2484,0.4397,0.1124,0.6408,0.4412
...,...,...,...,...,...,...,...,...,...,...
183,to,183,0.4818,0.4647,-0.8540,-1.5642,-0.1572,0.9397,-0.1491,-0.1436
184,take,184,3.7251,1.7327,-0.7423,0.1446,0.3173,0.6298,-0.4704,0.2151
185,out,185,-0.4130,-1.8310,-0.0338,0.3661,-0.0235,-0.4511,-2.2000,1.4692
186,of,186,-1.6804,1.1525,1.8086,-1.5669,1.3986,0.2751,-0.3336,-0.6778


## 2a) Learned Positional Embeddings

Unlike sinusoidal encodings—which use fixed mathematical waves—**learned positional embeddings**
are simply trainable vectors assigned to each position in a sequence.

A Transformer stores these in a matrix shaped:

(max_seq_len, d_model)

- Each **row** corresponds to a position (0, 1, 2, …)  
- Each **row is a learnable vector**  
- All rows are trained by gradient descent, just like token embeddings

During training, the model gradually adjusts these vectors to represent whatever
positional patterns are most helpful for understanding language.

In this cell:

1. We create a matrix of learned positional embeddings  
   (randomly initialized, as they would be at the start of training)

2. We assign one positional embedding to each token based on its index  
   (position 0 → first vector, position 1 → second vector, …)

3. We display the full lookup table:
   - **token** — the actual token string  
   - **position** — its index in the sequence  
   - **pos_dim_i** — the learned embedding values for that position  

This lets us see exactly how a Transformer stores and retrieves positional
information when using *learned* position embeddings.

In [10]:
import torch
import pandas as pd
from IPython.display import display, HTML

# -------------------------------------------
# Re-use your existing tokens and setup
# -------------------------------------------
n = len(tokens)      # number of tokens
d_model = 8

# -------------------------------------------
# Create a learned positional embedding matrix
# (what a Transformer learns during training)
# -------------------------------------------
torch.manual_seed(42)   # reproducible demo
pos_embedding_matrix = torch.randn(n, d_model)

# -------------------------------------------
# Build positional embedding table
# -------------------------------------------
df_pos = pd.DataFrame(
    pos_embedding_matrix.numpy().round(4),
    columns=[f"pos_dim_{i}" for i in range(d_model)]
)

df_pos.insert(0, "position", range(n))
df_pos.insert(1, "token", tokens)

# -------------------------------------------
# Display positional embeddings
# -------------------------------------------
display(HTML("<h3>2) Learned Positional Embeddings (trainable parameters)</h3>"))
display(df_pos)

,position,token,pos_dim_0,pos_dim_1,pos_dim_2,pos_dim_3,pos_dim_4,pos_dim_5,pos_dim_6,pos_dim_7
0,0,Alice,1.9269,1.4873,0.9007,-2.1055,0.6784,-1.2345,-0.0431,-1.6047
1,1,was,-0.7521,1.6487,-0.3925,-1.4036,-0.7279,-0.5594,-0.7688,0.7624
2,2,beginning,1.6423,-0.1596,-0.4974,0.4396,-0.7581,1.0783,0.8008,1.6806
3,3,to,1.2791,1.2964,0.6105,1.3347,-0.2316,0.0418,-0.2516,0.8599
4,4,get,-1.3847,-0.8712,-0.2234,1.7174,0.3189,-0.4245,0.3057,-0.7746
...,...,...,...,...,...,...,...,...,...,...
183,183,to,1.3401,1.5109,-1.1985,0.3767,-0.2511,-0.5277,-0.4770,-0.5625
184,184,take,-1.0563,0.2413,0.1828,0.6247,-0.7940,-0.6748,-0.3877,0.4497
185,185,out,0.3726,-1.9105,0.2609,1.4178,0.6738,1.4666,-1.1077,-0.7444
186,186,of,1.0189,-1.8317,0.2806,0.6909,0.7184,-0.5719,-0.4666,0.1018


## 2b) How Learned Positional Embeddings Are Trained

Just like token embeddings, **positional embeddings are updated through training**.
They begin as random vectors, but gradually change as the model sees millions of
sentences.

Whenever the model makes a prediction:

1. It compares its output to the correct next token.
2. It computes a **loss** (a measure of how wrong it was).
3. The model backpropagates the error signal through the network.
4. The positional embedding for each position receives a small update.

This process teaches the model how each position is typically used in real text:

- early positions often start clauses
- middle positions hold the “core” content
- later positions often close the sentence

After enough training, each position’s vector captures patterns that help
self-attention understand the structure of sentences.

We will explore this training process in detail later, but this gives an
intuition for *how* positional embeddings become meaningful.

## 3) Combining Token and Positional Embeddings

Now that we have:

- **Token embeddings** — learned representations of word meaning  
- **Positional embeddings** — learned representations of where each token appears  

the Transformer performs a single, simple operation:

### **Add them together**

For each token at position *i*:

combined_embedding[i] = token_embedding[i] + positional_embedding[i]

This produces a new vector that contains **both** kinds of information:

- the *semantics* of the token  
- the *position* of the token in the sequence  

These combined vectors form the actual input to the **first Transformer layer**.

---

### Why addition?

Addition is the simplest way to merge two sources of information while keeping
their dimensions identical.

- No extra parameters  
- No change in shape  
- Fast and easy for the model to learn with  

The model’s attention heads will later process these combined vectors to learn
patterns of **order**, **structure**, and **relationships** across the sequence.

---

### What the table shows

For each token you now see:

- its **string form**  
- its **position index**  
- the final **combined embedding** after adding token + positional embeddings  

This is exactly what a real Transformer uses as the starting point for all its
attention and reasoning.

In [9]:
import torch
import pandas as pd
from IPython.display import display, HTML

# -------------------------------------------
# Combine token + positional embeddings
# -------------------------------------------
combined_embeddings = fake_token_embeds + pos_embedding_matrix

# -------------------------------------------
# Build table
# -------------------------------------------
df_combined = pd.DataFrame(
    combined_embeddings.numpy().round(4),
    columns=[f"combined_dim_{i}" for i in range(d_model)]
)

df_combined.insert(0, "token", tokens)
df_combined.insert(1, "position", range(n))

# -------------------------------------------
# Display
# -------------------------------------------
display(HTML("<h3>3) Combined Token + Positional Embeddings</h3>"))
display(df_combined)

,token,position,combined_dim_0,combined_dim_1,combined_dim_2,combined_dim_3,combined_dim_4,combined_dim_5,combined_dim_6,combined_dim_7
0,Alice,0,0.8011,0.3349,0.6501,-2.5394,1.5271,-0.5425,-0.3591,-3.7199
1,was,1,-0.4299,0.3854,-0.0425,-1.0955,-0.6080,0.6782,0.3479,0.5152
2,beginning,2,0.2897,-1.8555,0.0693,1.2331,-0.1593,-0.4768,0.4594,3.5336
3,to,3,2.0293,0.7109,0.4371,1.5182,1.1577,1.6281,0.6947,0.0162
4,get,4,-1.9983,-0.8396,-0.7160,1.9658,0.7586,-0.3121,0.9465,-0.3334
...,...,...,...,...,...,...,...,...,...,...
183,to,183,1.8219,1.9756,-2.0525,-1.1876,-0.4084,0.4120,-0.6261,-0.7061
184,take,184,2.6688,1.9740,-0.5595,0.7693,-0.4767,-0.0450,-0.8581,0.6647
185,out,185,-0.0404,-3.7415,0.2270,1.7839,0.6503,1.0155,-3.3077,0.7248
186,of,186,-0.6615,-0.6792,2.0893,-0.8760,2.1170,-0.2968,-0.8002,-0.5760


## Next Step: Into Self-Attention

Once we have the **final embeddings** (token meaning + positional encoding), the Transformer sends them into three learned linear projections to create:

- **Queries (Q)**
- **Keys (K)**
- **Values (V)**

These are computed as:

$Q = XW_Q \;,\quad K = XW_K \;,\quad V = XW_V$

where $X$ is our final embedding matrix.

Q, K, and V are then used in **scaled dot-product attention**, which tells the model **how strongly each token should attend to every other token**.

This is the step where the Transformer starts to build meaningful relationships between words.